# 02. reverse B-S回帰用の多国間パネル構築

「高所得クラブ」（一人当たりGDP3万ドル以上。タックスヘイブン・金融センター、産油国・資源国、
小国、統計の信頼性に疑問のある国を事前に除外）を定義し、2005〜2023年の多国・多年パネルを作る。
2023年単年の横断面に加え、年をプールしたパネルと、複数の所得閾値での頑健性も確認する。


In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed"

START_YEAR = 2005
END_YEAR = 2023
INCOME_THRESHOLD = 30000  # 一人当たりGDP、2021年米ドル

# 事前に定めた除外リスト（論文の注1）。
# タックスヘイブン・金融センター型（IRL, LUX, SGP, MAC, MLT, CYM, BMU, VGB, ABW, SXM, AIA, MSR）、
# 産油国・資源国（QAT, ARE, KWT, BRN, SAU, OMN, BHR, KAZ, TKM）、
# 小島嶼国で一人当たりGDPが歪みやすい国（KNA, ATG, SYC, TCA, GUY, TTO, BHS）、
# 非市場経済・統計の信頼性に留保がある国（BLR, RUS）。
EXCL = {
    "IRL", "LUX", "QAT", "ARE", "KWT", "BRN", "SGP", "MAC", "MLT", "CYM", "BMU", "VGB",
    "ABW", "SXM", "AIA", "MSR", "KNA", "ATG", "SYC", "TCA", "GUY", "TTO", "BHS", "SAU",
    "OMN", "BHR", "KAZ", "TKM", "BLR", "RUS",
}
FOCUS_COUNTRIES = {"JPN": "日本", "KOR": "韓国", "TWN": "台湾"}

pwt = pd.read_csv(PROCESSED_DIR / "pwt_relative_price_income.csv")
print(f"読み込んだPWT: {pwt['countrycode'].nunique()}か国、{len(pwt):,}行")

読み込んだPWT: 185か国、11,201行


## 1. 高所得クラブパネルの構築（2005〜2023年、プール）

一人当たりGDP3万ドル以上・除外リスト外・`relp`が正の観測だけを残す。

In [2]:
pool = pwt[
    pwt["year"].between(START_YEAR, END_YEAR)
    & (pwt["rgdpe_pc"] >= INCOME_THRESHOLD)
    & (~pwt["countrycode"].isin(EXCL))
    & (pwt["relp"] > 0)
    & (pwt["reli"] > 0)
].copy()

print(f"高所得クラブパネル（プール）: {pool['countrycode'].nunique()}か国、{len(pool):,}国年")
print(f"対象国: {sorted(pool['countrycode'].unique())}")

for cc, name in FOCUS_COUNTRIES.items():
    in_pool = cc in pool["countrycode"].unique()
    print(f"  {name}（{cc}）は高所得クラブに含まれるか: {in_pool}")

高所得クラブパネル（プール）: 41か国、621国年
対象国: ['AUS', 'AUT', 'BEL', 'BGR', 'CAN', 'CHE', 'CYP', 'CZE', 'DEU', 'DNK', 'ESP', 'EST', 'FIN', 'FRA', 'GBR', 'GNQ', 'GRC', 'HKG', 'HRV', 'HUN', 'ISL', 'ISR', 'ITA', 'JPN', 'KOR', 'LTU', 'LVA', 'MYS', 'NLD', 'NOR', 'NZL', 'PAN', 'POL', 'PRT', 'ROU', 'SVK', 'SVN', 'SWE', 'TUR', 'TWN', 'USA']
  日本（JPN）は高所得クラブに含まれるか: True
  韓国（KOR）は高所得クラブに含まれるか: True
  台湾（TWN）は高所得クラブに含まれるか: True


## 2. 2023年単年の横断面

2023年単年の横断面（世界全体と高所得クラブ）も別途保存しておく。


In [3]:
cross_section_2023 = pwt[
    (pwt["year"] == 2023) & (pwt["relp"] > 0) & (pwt["reli"] > 0)
].copy()
high_income_2023 = cross_section_2023[
    (cross_section_2023["rgdpe_pc"] >= INCOME_THRESHOLD) & (~cross_section_2023["countrycode"].isin(EXCL))
].copy()

print(f"2023年・世界全体: {len(cross_section_2023)}か国")
print(f"2023年・高所得クラブ: {len(high_income_2023)}か国")

2023年・世界全体: 185か国
2023年・高所得クラブ: 40か国


## 3. 所得閾値の頑健性確認用データ

閾値（2.5万／3万／3.5万ドル）を変えたときの対象国数を確認する
（実際の回帰・頑健性チェックは03・05で行う）。

In [4]:
for threshold in (25000, 30000, 35000):
    subset = pwt[
        (pwt["year"] == 2023) & (pwt["rgdpe_pc"] >= threshold)
        & (~pwt["countrycode"].isin(EXCL)) & (pwt["relp"] > 0)
    ]
    print(f"閾値{threshold:,}ドル: {len(subset)}か国")

閾値25,000ドル: 46か国
閾値30,000ドル: 40か国
閾値35,000ドル: 37か国


## 4. 保存

In [5]:
pool.to_csv(PROCESSED_DIR / "ppp_panel_pooled_2005_2023.csv", index=False)
cross_section_2023.to_csv(PROCESSED_DIR / "ppp_cross_section_2023_world.csv", index=False)
high_income_2023.to_csv(PROCESSED_DIR / "ppp_cross_section_2023_highincome.csv", index=False)

print("保存先:")
print(f"  {(PROCESSED_DIR / 'ppp_panel_pooled_2005_2023.csv').relative_to(BASE_DIR)}")
print(f"  {(PROCESSED_DIR / 'ppp_cross_section_2023_world.csv').relative_to(BASE_DIR)}")
print(f"  {(PROCESSED_DIR / 'ppp_cross_section_2023_highincome.csv').relative_to(BASE_DIR)}")

保存先:
  data\processed\ppp_panel_pooled_2005_2023.csv
  data\processed\ppp_cross_section_2023_world.csv
  data\processed\ppp_cross_section_2023_highincome.csv


## まとめ

3つのパネル／横断面を保存した：

- `ppp_panel_pooled_2005_2023.csv` — 高所得クラブ、2005〜2023年プール（03の主分析用）
- `ppp_cross_section_2023_world.csv` — 2023年・世界全体（185か国）
- `ppp_cross_section_2023_highincome.csv` — 2023年・高所得クラブ単年

次は `03_reverse_bs_regression.ipynb` で、`log(relp) ~ log(reli)` の回帰
（Penn effect／reverse B-S回帰）を行い、日本・台湾・韓国の残差を計算する。